# Logistic Regression Baseline - CLINC150

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import copy
import time
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = False
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda


15250 / 3100 / 5500, 151 classes (full)


In [2]:
MAX_FEATURES = 5000

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), min_df=2, max_df=0.95)

train_features = vectorizer.fit_transform(train_texts).toarray()
val_features = vectorizer.transform(val_texts).toarray()
test_features = vectorizer.transform(test_texts).toarray()

print(f"TF-IDF dim: {train_features.shape[1]}")

TF-IDF dim: 5000


In [3]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


BATCH_SIZE = 128

train_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TextDataset(test_features, test_labels), batch_size=BATCH_SIZE, shuffle=False)

In [4]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)


model = LogisticRegression(train_features.shape[1], num_classes).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} params")

755,151 params


In [5]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            total_loss += criterion(outputs, labels).item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

In [6]:
LEARNING_RATE = 0.01
NUM_EPOCHS = 50
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_f1, best_model_state, patience_counter, best_epoch = 0, None, 0, 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_preds, vl_true = evaluate(model, val_loader, criterion)
    vl_f1 = f1_score(vl_true, vl_preds, average='macro', zero_division=0)
    vl_acc = accuracy_score(vl_true, vl_preds)

    mark = ""
    if vl_f1 > best_val_f1:
        best_val_f1, best_model_state, best_epoch, patience_counter = vl_f1, copy.deepcopy(model.state_dict()), epoch + 1, 0
        mark = " *"
    else:
        patience_counter += 1

    print(f"{epoch+1:2d}/{NUM_EPOCHS}  tr_loss={tr_loss:.4f} tr_acc={tr_acc:.1f}%  vl_loss={vl_loss:.4f} vl_acc={vl_acc*100:.1f}% vl_f1={vl_f1:.4f}{mark}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping. Best epoch: {best_epoch}, F1: {best_val_f1:.4f}")
        break

training_time = time.time() - start_time
print(f"Time: {training_time:.1f}s")

 1/50  tr_loss=4.0126 tr_acc=70.3%  vl_loss=3.1288 vl_acc=83.0% vl_f1=0.8430 *
 2/50  tr_loss=2.0181 tr_acc=95.1%  vl_loss=1.8883 vl_acc=85.5% vl_f1=0.8614 *
 3/50  tr_loss=1.0056 tr_acc=96.7%  vl_loss=1.3302 vl_acc=86.5% vl_f1=0.8718 *
 4/50  tr_loss=0.5893 tr_acc=97.4%  vl_loss=1.0659 vl_acc=86.7% vl_f1=0.8734 *
 5/50  tr_loss=0.3989 tr_acc=97.9%  vl_loss=0.9200 vl_acc=87.2% vl_f1=0.8790 *
 6/50  tr_loss=0.2961 tr_acc=98.2%  vl_loss=0.8295 vl_acc=87.4% vl_f1=0.8814 *
 7/50  tr_loss=0.2301 tr_acc=98.4%  vl_loss=0.7658 vl_acc=87.8% vl_f1=0.8848 *
 8/50  tr_loss=0.1878 tr_acc=98.6%  vl_loss=0.7211 vl_acc=87.8% vl_f1=0.8856 *
 9/50  tr_loss=0.1572 tr_acc=98.8%  vl_loss=0.6889 vl_acc=87.9% vl_f1=0.8868 *
10/50  tr_loss=0.1337 tr_acc=98.9%  vl_loss=0.6621 vl_acc=87.8% vl_f1=0.8861
11/50  tr_loss=0.1163 tr_acc=99.0%  vl_loss=0.6410 vl_acc=87.8% vl_f1=0.8858
12/50  tr_loss=0.1021 tr_acc=99.1%  vl_loss=0.6238 vl_acc=87.9% vl_f1=0.8869 *
13/50  tr_loss=0.0910 tr_acc=99.2%  vl_loss=0.6104 vl_ac

In [7]:
model.load_state_dict(best_model_state)
_, test_preds, test_true = evaluate(model, test_loader, criterion)

acc = accuracy_score(test_true, test_preds)
p = precision_score(test_true, test_preds, average='macro', zero_division=0)
r = recall_score(test_true, test_preds, average='macro', zero_division=0)
f1_m = f1_score(test_true, test_preds, average='macro', zero_division=0)
f1_w = f1_score(test_true, test_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

baseline_lr_preds = [int(x) for x in test_preds]

Accuracy:   0.8033
Precision:  0.8119
Recall:     0.8858
F1 macro:   0.8414
F1 weighted:0.7931


In [8]:
results = {
    "model_type": "LogisticRegression",
    "optimization": "baseline",
    "best_epoch": best_epoch,
    "training_time_seconds": round(training_time, 2),
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "predictions": baseline_lr_preds,
    "hyperparameters": {
        "max_features": MAX_FEATURES,
        "ngram_range": [1, 2],
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": NUM_EPOCHS,
        "early_stopping_patience": PATIENCE
    }
}

with open('results/results_lr_baseline.json', 'w') as f:
    json.dump(results, f, indent=4)

print("Saved: results/results_lr_baseline.json")

Saved: results/results_lr_baseline.json
